# Module 5 — Inverse Kinematics
## Unit 7 — Verifying and Connecting to Perception
### Lesson 7.4 — Verifying and Connecting to Perception (Unit 7 Recap)

*Physical AI Curriculum · hands-on notebook. Run **Kernel → Restart & Run All**.*


## Unit 7 synthesis
Perceive → base frame → solve → verify → select, end to end.


In [ ]:
import numpy as np

def fk_two_link(t1, t2, L1, L2):
    return np.array([L1*np.cos(t1)+L2*np.cos(t1+t2), L1*np.sin(t1)+L2*np.sin(t1+t2)])

def jacobian_2link(t1, t2, L1, L2):
    s1,s12=np.sin(t1),np.sin(t1+t2); c1,c12=np.cos(t1),np.cos(t1+t2)
    return np.array([[-L1*s1-L2*s12,-L2*s12],[L1*c1+L2*c12,L2*c12]])

def ik_2link_closed(x, y, L1, L2):
    c2=(x*x+y*y-L1*L1-L2*L2)/(2*L1*L2)
    if c2<-1-1e-9 or c2>1+1e-9: return []
    c2=max(-1.0,min(1.0,c2)); sols=[]
    for sign in (+1.0,-1.0):
        s2=sign*np.sqrt(max(0.0,1-c2*c2)); t2=np.arctan2(s2,c2)
        t1=np.arctan2(y,x)-np.arctan2(L2*np.sin(t2),L1+L2*np.cos(t2))
        sols.append((t1,t2))
        if abs(s2)<1e-6: break
    return sols

def reachable(x, y, L1, L2, tol=1e-9):
    r=np.hypot(x,y); return abs(L1-L2)-tol<=r<=L1+L2+tol

def ik_numerical(target, theta0, L1, L2, alpha=1.0, tol=1e-6, max_iter=100):
    theta=np.array(theta0,float); target=np.array(target,float)
    for it in range(max_iter):
        e=target-fk_two_link(theta[0],theta[1],L1,L2)
        if np.linalg.norm(e)<tol: return theta
        J=jacobian_2link(theta[0],theta[1],L1,L2)
        theta=theta+alpha*np.linalg.pinv(J)@e
    return theta

def norm180(a): return (a+180.0)%360.0-180.0

T=np.array([[1,0,0.1],[0,1,0.0],[0,0,1.0]])
L1,L2=0.4,0.3
# a fruit becomes a verified configuration
p_base=(T@np.array([0.4,0.2,1.0]))[:2]
cand=ik_2link_closed(*p_base,L1,L2)[0]
res=np.linalg.norm(p_base-fk_two_link(cand[0],cand[1],L1,L2))
print("fruit -> base", np.round(p_base,3), "-> verified residual", f"{res:.2e}")


## Check your work


In [ ]:
p_base=(T@np.array([0.4,0.2,1.0]))[:2]
cand=ik_2link_closed(*p_base,L1,L2)[0]
assert np.linalg.norm(p_base-fk_two_link(cand[0],cand[1],L1,L2))<1e-9
print("All checks passed.")
